# 🧪 Feature Selection Experiment — v2
## Student Performance Prediction (Pass / Fail)

### Objective
Predict whether a student will PASS or FAIL using only 5–6 minimal inputs that are realistic for Indian college students filling a Google Form.

### Feature Mapping (Dataset → Real-World)

| Dataset Column | Mapped Feature | Description |
| :--- | :--- | :--- |
| G2 | previous_sem_cgpa | CGPA of the immediately previous semester (scaled 0–10) |
| G1 | previous_to_previous_sem_cgpa | CGPA two semesters ago (scaled 0–10) |
| failures | number_of_backlogs | Count of past subject failures |
| absences | attendance_percentage | Derived attendance % (inverse of absences) |
| studytime | studytime | Weekly study hours (1–4 scale) |
| goout | goout | Going out frequency (1–5 scale) — optional |

### Target
- **PASS = 1** if G3 ≥ 10 (i.e. CGPA ≥ 5.0)
- **FAIL = 0** if G3 < 10

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay)

print("\u2705 All libraries loaded successfully!")

### Step 2: Load & Combine Datasets
Two CSV files from the UCI Student Performance dataset:
1. `student-mat.csv` — Math course (395 students)
2. `student-por.csv` — Portuguese course (649 students)

We combine them for a larger training pool (~1044 students).

In [ ]:
# Load datasets
math_df = pd.read_csv('../data/student-mat.csv', sep=';')
por_df  = pd.read_csv('../data/student-por.csv', sep=';')

print(f"Math dataset:       {math_df.shape[0]} students x {math_df.shape[1]} columns")
print(f"Portuguese dataset: {por_df.shape[0]} students x {por_df.shape[1]} columns")

# Combine
df_raw = pd.concat([math_df, por_df], ignore_index=True)
print(f"\nCombined dataset:   {df_raw.shape[0]} students x {df_raw.shape[1]} columns")
df_raw.head()

In [ ]:
# Feature engineering CGPA Mapping
df = df_raw.copy()
df['previous_sem_cgpa'] = df['G2'] / 2.0
df['previous_to_previous_sem_cgpa'] = df['G1'] / 2.0
df['number_of_backlogs'] = df['failures']
df['attendance_percentage'] = (100 - df['absences'] * 1.5).clip(lower=0, upper=100)
df['target'] = (df['G3'] >= 10).astype(int)

print("\u2705 Feature engineering complete!")

In [ ]:
FINAL_FEATURES = ['previous_sem_cgpa', 'previous_to_previous_sem_cgpa', 'number_of_backlogs', 'attendance_percentage', 'studytime', 'goout']
X_final = df[FINAL_FEATURES]
y_final = df['target']

X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.2, random_state=42, stratify=y_final)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes':         GaussianNB(),
    'SVM':                 SVC(kernel='rbf', probability=True, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=4, random_state=42, use_label_encoder=False, eval_metric='logloss', verbosity=0),
    'AdaBoost':            AdaBoostClassifier(n_estimators=100, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_scaled))
    print(f"{name}: {acc*100:.2f}%")